In [37]:
!yt-dlp -f m4a -o "./youtube_audios/%(title)s.%(ext)s" "https://youtu.be/t9ugPGO3_RM?si=2fank3nSloficzLX"


[youtube] Extracting URL: https://youtu.be/t9ugPGO3_RM?si=2fank3nSloficzLX
[youtube] t9ugPGO3_RM: Downloading webpage
[youtube] t9ugPGO3_RM: Downloading ios player API JSON
[youtube] t9ugPGO3_RM: Downloading m3u8 information
[info] t9ugPGO3_RM: Downloading 1 format(s): 140
[download] ./youtube_audios/고든램지 풀드포크 ： 육즙 대폭발! 바베큐도 고든램지가 하면 다르다 (Gordon Ramsay's Smoky Pulled Pork with Chipotle Mayonnaise).m4a has already been downloaded
[download] 100% of   11.78MiB


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
import librosa

# 디바이스 및 데이터 타입 설정
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 모델 로드
model_id = "openai/whisper-large-v3"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-large-v3", torch_dtype=torch.float16, use_safetensors=True, device_map='auto'
)

processor = AutoProcessor.from_pretrained(model_id)

# 자동 음성 인식 파이프라인 설정
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,  # 30초 단위로 오디오를 분할
    batch_size=16,
    return_timestamps=True,
    device=device,
)

# 오디오 파일 경로
audio_path = "./youtube_audios/고든램지 풀드포크 ： 육즙 대폭발! 바베큐도 고든램지가 하면 다르다 (Gordon Ramsay's Smoky Pulled Pork with Chipotle Mayonnaise).m4a"

# librosa를 사용하여 오디오 파일 읽기
audio, sr = librosa.load(audio_path, sr=16000)

# librosa로 로드한 오디오를 파이프라인에 직접 사용 가능하도록 변환
# 'sampling_rate'가 필요하기 때문에 librosa에서 추출한 sr을 사용
result = pipe({"array": audio, "sampling_rate": sr})

# 결과 출력
print(result["text"])

# 결과를 파일로 저장
output_file = "./transcripts/냄비뚜껑 훈제 삼겹살.txt"
with open(output_file, 'w', encoding='utf-8') as file:
    file.write(result["text"])

print(f"Transcript saved to {output_file}")


In [39]:
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain.llms import HuggingFacePipeline


In [41]:
# 텍스트 파일 경로
output_file = "./transcripts/냄비뚜껑 훈제 삼겹살.txt"

# 텍스트 파일을 읽기 모드로 열기
with open(output_file, 'r', encoding='utf-8') as file:
    # 파일 내용 읽기
    text = file.read()

# 파일 내용을 출력
print("Loaded Transcript:")
print(text)

Loaded Transcript:
 오늘 준비한 고기부터 보시죠 코스트코에서 산 돼지 목살입니다 언제나 말씀드리듯이 목살 원육은 코스트코가 최고입니다 원래는 이걸 한 덩이로 2만원대에 팔았었거든요 오늘은 가보니까 두 덩이 묶어서 4만원대에 팔더라고요 원래 풀드포크는 버트라고 하는 돼지 어깨 부위로 만듭니다 하지만 한국에서는 버트 커팅을 거의 구할 수 없기 때문에 저처럼 목살, 목심을 사용하시면 되겠습니다 풀드포크는 굳이 지방 손질을 할 필요가 없습니다 비계가 녹으면서 살코기 쪽의 육즙을 보충하기 때문이죠 이렇게 핏물을 잘 닦기만 하면 고기 준비가 완료됩니다 육식맨 채널 첫 영상이 오븐 풀드포크였고요 수비드 풀드포크 영상도 큰 사랑을 받았는데요 오늘은 램지 형님을 모셔서 풀드포크를 만들어봅니다 거의 천만 뷰가 나온 초대박 비디오로 98년생 맏딸 레간과 함께 만들기 때문에 더 행복해 보이는 영상입니다 고든 램지 모든 영상은 계량이나 비율을 정확히 공개하지 않아서 20회 이상 돌려보며 제가 직접 계량을 잡았습니다 풀드포크에 곁들이는 치폴레 마요네즈 소스 심지어 브로콜리 샐러드까지 보이는 그대로 만들어서 램지 패밀리 같이 만찬을 즐겨보겠습니다 먼저 목살에 펴바를 마리네이드 소스를 만듭니다 상세 레시피는 영상 정보란에 써놓겠습니다 시작은 마늘입니다 고든 램지는 겨우 두 개 쓰는 거 같던데 저는 한국인답게 양껏 다져놨고요 오늘의 핵심 재료 스모크 파프리카 파우더입니다 해외 직구 안 하고 살 수 있는 두 가지 제품을 다 사봤어요 오늘 요리 정식 제목이 스모키 풀드 포크예요 그러니까 그냥 파프리카 말고 꼭 스모크로 사시길 바랍니다. 램지처럼 아주 왕창 넣겠습니다. 다음은 설탕입니다. 원래 서양 통고기 요리에는 설탕이 많이 들어갑니다. 도리어 고든 램지가 적은 편이에요. 황설탕이나 흑설탕을 쓰시면 됩니다. 우리 램지 형님과는 뗄 수 없는 솔트앤페퍼도 넣었고요. 허브, 타임을 넣었습니다. 고든 램지는 생타임의 잎을 넣던데요. 마리네이드 소스에 생은 너무 아까워서 건타임을 왕창 

In [42]:
# 텍스트를 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=150)
splits = text_splitter.split_text(text)

In [43]:
model = AutoModelForCausalLM.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", torch_dtype=torch.float16, trust_remote_code=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", trust_remote_cote=True)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [23]:
# 파이프라인 객체 생성
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=1024)
    
# HuggingFacePipeline으로 모델 래핑
llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [15]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cuda"})
vectordb = FAISS.from_texts(splits, embeddings)

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [44]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectordb.as_retriever(),
    return_source_documents=True   
)


In [45]:
# 질문을 하세요!
query = "전체적인 레시피에 대해 알려주세요"
answer = qa_chain({"query": query})

KeyboardInterrupt: 

In [34]:
print(answer['result'])

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

비계 부분 역시 쌈장으로 도포했고요. 다시 한번 팽팽하게 단단하게 실을 묶으면 K시즈닝까지 마친 진짜 고기냄뚜 완성입니다 이제 물을 끓일 건데요 오늘의 광고주님 아이오닉5 아이오닉5의 멀티탭과 커피포트를 연결해서 물을 끓일 거예요 여기서 끝이 아니죠 인덕션까지 사용할 겁니다 아시다시피 인덕션은 전기 먹는 하마 수준인데요 전기차의 전기로 얼마나 오래 요리할 수 있는지 시험해볼 거예요 오늘은 고기를 삶지 않고 찔 거기 때문에 물은 조금만 채웠고요 수증기만으로도 존재감을 뿜어내는 강력한 향신료들을 넣을 거예요 마늘 한 주먹, 통후추 30알, 팔각 3개, 월계수잎 3장을 넣었습니다 고기가 신선하다면 이거 다 필요 없어요 맹물도 돼요 이제 고기 뚜껑을 닫아 이대로 2시간 익히겠습니다 예상치도 못했는데 팔각형이 예쁘네요 아주 먹음직스럽게 수육이 삶아졌습니다 고기 중심부도 70도 이상으로 완전히 다 익었습니다 엄청난 된장 수육이 만들어졌지만 솔직히 놀랍진 않죠 무수분 수육이라고 수증기에 찌는 수육은 꽤 대중적이거든요 그래서 지금부터 수추로 업그레이드 시킬 거예요 수추에 불을 붙여 하얗게 만든 뒤 호일 접시에 올려 담아놨어요 그 다음 이거는 훈연칩인데 물에 불려 놨습니다 이것도 연기가 나긴 하는데 훈연칩이 있어야지 좀 맛있는 연기가 나요 연기가 뿜어서 나기도 하고 물에 불리지 않으면 연기가 잘 안 나더라고요 그냥 장작처럼 돼가지고 버리는 냄비라면 숯을 그냥 넣어도 되는데 저는 소중한 무쇠 냄비라서 호일 접시를 씁니다 숯 위에 훈연칩을 올리자마자 아주 향긋한 연기가 뿜어져 나오기 시작합니다 뚜껑을 아주 약간의 틈만 남긴 채로 덮으면 되는데요 원래 뚜껑을 닫으면 불이 꺼져야 정상이잖아요 근